# Lecture 5 — Class Exercise
## Distribution Charts: Airbnb London

> **Push to:** `week05/lecture05_exercise.ipynb`

**Rules:**
1. Cap price outliers at 95th percentile — annotate this
2. Every chart has a **median/mean reference line** with annotation
3. Insight title names the distribution shape or key finding
4. Colour has meaning — don't use colour just for decoration

---


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Airbnb London Listings

df = pd.read_csv('../data/airbnb_london.csv')
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))


Loaded: 2500 listings
        price  minimum_nights  number_of_reviews  availability_365  \
count  2500.0          2500.0             2500.0            2500.0   
mean    148.6            14.8              147.9             183.7   
std     110.9             8.4               86.3             105.5   
min      20.5             1.0                0.0               0.0   
25%      71.7             8.0               74.0              92.0   
50%     117.5            15.0              145.0             182.0   
75%     188.9            22.0              222.2             277.0   
max    1032.4            29.0              299.0             364.0   

       reviews_per_month  
count             2500.0  
mean                 2.0  
std                  2.0  
min                  0.0  
25%                  0.6  
50%                  1.4  
75%                  2.8  
max                 15.2  


In [2]:
p95 = df['price'].quantile(0.95)
df_cap = df[df['price'] <= p95]
print(f"95th percentile price: £{p95:.0f}")
print(df_cap.groupby('room_type')['price'].describe().round(1))


95th percentile price: £373
                  count   mean   std   min    25%    50%    75%    max
room_type                                                             
Entire home/apt  1251.0  176.3  75.7  28.0  119.6  163.4  223.5  372.6
Private room      942.0   87.3  39.5  20.9   59.0   78.6  106.0  277.9
Shared room       182.0   46.3  14.1  20.5   36.8   44.1   54.3   92.8


## Task 1 — Histogram: price by room type (overlapping distributions)

**What to build:** A histogram showing price distributions for **Entire home/apt vs Private room** (exclude Shared room — too few observations) overlaid on the same chart.

**Requirements:**
- Both room types on the same chart (use `color='room_type'`)
- `barmode='overlay'` with `opacity=0.6` so both distributions are visible
- A vertical line for the median of EACH room type, differently coloured
- Insight title comparing the two distributions

> 💡 `df_cap[df_cap['room_type'].isin(['Entire home/apt','Private room'])]`


In [4]:
# Task 1
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv('../data/airbnb_london.csv')

# Cap extreme prices for a cleaner distribution view
PRICE_CAP = 500
df_cap = df[df['price'] <= PRICE_CAP].copy()

# Filter to the two room types only
room_types = ['Entire home/apt', 'Private room']
df_cap = df_cap[df_cap['room_type'].isin(room_types)]

COLOR = {
    'Entire home/apt': '#E63946',
    'Private room':    '#457B9D'
}

# Medians for vertical lines
medians = df_cap.groupby('room_type')['price'].median()

fig = go.Figure()

# ── Overlaid histograms ───────────────────────────────────────────────────────
for room in room_types:
    d = df_cap[df_cap['room_type'] == room]['price']
    fig.add_trace(go.Histogram(
        x=d,
        name=room,
        marker_color=COLOR[room],
        opacity=0.6,
        xbins=dict(start=0, end=PRICE_CAP, size=10),
        hovertemplate=f"<b>{room}</b><br>Price: $%{{x}}<br>Count: %{{y}}<extra></extra>"
    ))

# ── Median vertical lines ─────────────────────────────────────────────────────
for room in room_types:
    med = medians[room]
    fig.add_vline(
        x=med,
        line_color=COLOR[room],
        line_width=2,
        line_dash='dash',
        annotation=dict(
            text=f"<b>{room} median</b><br>${med:.0f}",
            font=dict(color=COLOR[room], size=11),
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor=COLOR[room],
            borderwidth=1,
            borderpad=4,
        ),
        annotation_position='top right' if room == 'Entire home/apt' else 'top left'
    )

fig.update_layout(
    barmode='overlay',
    title=dict(
        text=(
            f"<b>Entire homes cost ~${medians['Entire home/apt']:.0f}/night at median — "
            f"nearly double Private rooms at ~${medians['Private room']:.0f}</b><br>"
            "<sup>Price distribution of Airbnb listings (capped at $500) · 10-dollar bins</sup>"
        ),
        font=dict(size=15, family='Arial'),
        x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='Price per Night (USD)',
        tickprefix='$',
        showgrid=True, gridcolor='#EEEEEE',
        range=[0, PRICE_CAP]
    ),
    yaxis=dict(
        title='Number of Listings',
        showgrid=True, gridcolor='#EEEEEE'
    ),
    legend=dict(
        title='Room Type',
        orientation='h',
        x=0.99, y=0.99,
        xanchor='right', yanchor='top',
        font=dict(size=12)
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=60, r=60, t=110, b=60),
    height=520, width=950
)

fig.show()


## Task 2 — Box plot: listing activity by borough

**What to build:** A **horizontal box plot** comparing listing activity (reviews per month) across London boroughs — reviews per month is a proxy for how frequently a listing is booked.

**Requirements:**
- Horizontal orientation (borough names are long)
- Sorted by median reviews per month (most active at top)
- Highlight the **two most active** boroughs in a different colour
- Outliers shown as individual points
- Insight title naming the two busiest boroughs

> 💡 Some listings have zero reviews — these are new or inactive listings. Filter them out with before plotting

In [7]:
# Task 2
# YOUR CODE HERE
# Task 2 — Box plot: listing activity by borough
 
df_task2 = df_cap[df_cap['reviews_per_month'] > 0].copy()
 
borough_order = (

    df_task2.groupby('neighbourhood')['reviews_per_month']

    .median()

    .sort_values(ascending=True)

    .index.tolist()

)

top2 = borough_order[-2:]
 
df_task2['activity_tier'] = df_task2['neighbourhood'].apply(

    lambda b: 'Top 2 Most Active' if b in top2 else 'Other Boroughs'

)
 
fig2 = px.box(

    df_task2,

    x='reviews_per_month',

    y='neighbourhood',

    color='activity_tier',

    color_discrete_map={'Top 2 Most Active': '#E91E63', 'Other Boroughs': '#B0BEC5'},

    orientation='h',

    points='outliers',

    category_orders={'neighbourhood': borough_order},

    labels={'reviews_per_month': 'Reviews per Month', 'neighbourhood': 'Borough'},

    title=f'{top2[-1]} & {top2[-2]} Lead London in Booking Activity',

)
 
fig2.update_layout(plot_bgcolor='white', paper_bgcolor='white')

fig2.update_xaxes(showgrid=True, gridcolor='#EEEEEE')

fig2.update_yaxes(showgrid=True, gridcolor='#EEEEEE')

fig2.show()
 